# 08 · Condition-blind cNMF program review

This notebook evaluates whether each cNMF program is a coherent state of its source lineage and whether its spatial pattern appears biological rather than driven by spillover, segmentation, tissue edges, or one section.

The review hides condition labels and replaces sample IDs with deterministic section aliases. Select one lineage and program, inspect the evidence panels, and save a decision only after enabling the explicit write switch.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import anndata as ad
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "spatial_workflow").is_dir():
    candidate = REPO_ROOT.parent
    if (candidate / "src" / "spatial_workflow").is_dir():
        REPO_ROOT = candidate
    else:
        raise RuntimeError("Start Jupyter from the spatial-workflow repository")
sys.path.insert(0, str(REPO_ROOT / "src"))

from spatial_workflow.cnmf import usage_columns
from spatial_workflow.cnmf_program_review import (
    build_review_decision,
    high_usage_gene_support,
    high_usage_neighbor_context,
    plot_blinded_spatial_usage,
    prepare_blinded_usage_obs,
    program_usage_qc_summary,
    select_high_usage_cells,
    upsert_review_decision,
)

CNMF_ROOT = REPO_ROOT / "results" / "ab_xenium" / "05_cnmf"
WHITELIST_PATH = (
    CNMF_ROOT / "program_review" / "cnmf_program_whitelist_draft.tsv"
)
METRICS_PATH = (
    CNMF_ROOT
    / "program_review"
    / "cnmf_program_review_metrics_condition_blind.tsv"
)
QUEUE_PATH = (
    CNMF_ROOT / "program_review" / "cnmf_program_review_queue_draft.tsv"
)
DECISION_PATH = (
    CNMF_ROOT / "program_review" / "human_review_decisions_draft.tsv"
)
MASTER_H5AD = (
    CNMF_ROOT
    / "integrated"
    / "ab_xenium_cellcharter_cnmf_selected_draft.h5ad"
)

LINEAGE_DIRECTORIES = {
    "astrocyte": "astrocyte",
    "oligodendrocyte": "oligodendrocyte",
    "microglia": "microglia_cluster_sub_all",
    "inhibitory_neuron": "inhibitory_neuron",
    "excitatory_neuron": "excitatory_neuron",
    "perivascular": "perivascular",
    "epd": "epd",
    "chp": "chp",
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## Program inventory

This section verifies the draft whitelist, its checksum, and the source usage objects. Counts are summarized by lineage and draft decision without exposing condition information.

In [ ]:
whitelist = pd.read_csv(
    WHITELIST_PATH, sep="\t", dtype=str, keep_default_na=False
)
whitelist_sha256 = file_sha256(WHITELIST_PATH)

inventory = (
    whitelist.groupby("lineage", sort=False)
    .agg(
        selected_k=("selected_k", "first"),
        raw_programs=("program", "size"),
        primary_programs=("primary_include", lambda s: int(s.eq("TRUE").sum())),
        sensitivity_total=("sensitivity_include", lambda s: int(s.eq("TRUE").sum())),
        excluded_programs=("decision", lambda s: int(s.eq("exclude").sum())),
    )
    .reset_index()
)
display(inventory)
print("whitelist SHA-256:", whitelist_sha256)

program_metrics = pd.read_csv(METRICS_PATH, sep="\t")
review_queue = pd.read_csv(QUEUE_PATH, sep="\t", keep_default_na=False)
if not program_metrics["metric_condition_blind"].astype(bool).all():
    raise RuntimeError("Program metric table is not marked condition-blind")

review_columns = [
    "review_order", "review_tier", "lineage", "program", "decision", "metric_flags"
]
display(review_queue[review_columns].head(25))

## Select a program for review

Choose one lineage–program pair and the fraction of highest-usage cells to inspect. The notebook uses an exact ranked-cell count and a configurable neighbor radius for contamination screening.

Run the remaining sections after changing these controls; all displays remain condition blind.

In [ ]:
REVIEW_ORDER = int(review_queue["review_order"].min())
matches = review_queue.loc[review_queue["review_order"].eq(REVIEW_ORDER)]
if len(matches) != 1:
    raise RuntimeError(f"Expected one review item for order {REVIEW_ORDER}; found {len(matches)}")
review_item = matches.iloc[0]
LINEAGE = str(review_item["lineage"])
PROGRAM = str(review_item["program"])

TOP_FRACTION = 0.05
NEIGHBOR_RADIUS_UM = 30.0
MAX_CONTEXT_FOCAL_CELLS = 2_000
MAX_CORRELATION_CELLS = 50_000

# None selects the blinded section containing the most high-usage cells.
SPATIAL_SECTION = None
SPATIAL_POINT_SIZE = 3.0

display(review_item.to_frame("value"))
print(f"Reviewing {REVIEW_ORDER}/{len(review_queue)}: {LINEAGE} / {PROGRAM}")

In [ ]:
if LINEAGE not in LINEAGE_DIRECTORIES:
    raise KeyError(f"Unknown lineage {LINEAGE!r}: {list(LINEAGE_DIRECTORIES)}")

lineage_root = CNMF_ROOT / LINEAGE_DIRECTORIES[LINEAGE]
manifest = json.loads((lineage_root / "run_manifest.json").read_text())
usage_adata = ad.read_h5ad(Path(manifest["paths"]["usage_h5ad"]), backed="r")
programs = usage_columns(usage_adata)
if PROGRAM not in programs:
    raise KeyError(f"{PROGRAM!r} is unavailable; choose one of {programs}")

review_obs, sample_aliases = prepare_blinded_usage_obs(
    usage_adata.obs, sample_key="sample_id", usage_cols=programs
)
assert "condition" not in review_obs
assert "sample_id" not in review_obs

program_row = whitelist.loc[
    whitelist["lineage"].eq(LINEAGE) & whitelist["program"].eq(PROGRAM)
]
if len(program_row) != 1:
    raise RuntimeError(
        f"Expected one whitelist row for {LINEAGE}/{PROGRAM}; found {len(program_row)}"
    )
program_row = program_row.iloc[0]

draft_columns = [
    "lineage", "program", "proposed_label", "category",
    "confidence", "decision", "rationale", "top_genes",
]
display(program_row[draft_columns].to_frame("draft_value"))

current_metrics = program_metrics.loc[
    program_metrics["lineage"].eq(LINEAGE)
    & program_metrics["program"].eq(PROGRAM)
]
if len(current_metrics) != 1:
    raise RuntimeError(
        f"Expected one metric row for {LINEAGE}/{PROGRAM}; found {len(current_metrics)}"
    )
metric_columns = [
    "sd_usage", "iqr_usage", "nonzero_fraction", "n_top_sections",
    "largest_section_fraction", "effective_top_sections", "spatial_knn_enrichment",
    "spearman_nCount_Xenium", "spearman_nFeature_Xenium",
    "max_abs_pairwise_spearman_program", "max_abs_pairwise_spearman",
    "mean_top_gene_detection_fraction", "median_top_gene_log2_mean_count_ratio",
]
display(current_metrics.iloc[0][metric_columns].to_frame("condition_blind_metric"))
print(
    f"{LINEAGE}: {usage_adata.n_obs:,} cells, {len(programs)} programs, "
    f"{review_obs['blinded_sample'].nunique()} blinded sections"
)

## Usage distribution across blinded sections

Inspect the overall usage distribution and how strongly high-usage cells concentrate in individual blinded sections. Section concentration is a diagnostic flag, not an automatic exclusion criterion.

In [ ]:
qc_summary = program_usage_qc_summary(
    review_obs, usage_cols=programs, top_fraction=TOP_FRACTION
)
display(qc_summary.loc[qc_summary["program"].eq(PROGRAM)])

usage_plot = review_obs[["blinded_sample", PROGRAM]].copy()
usage_plot[PROGRAM] = pd.to_numeric(usage_plot[PROGRAM], errors="coerce")
figure = px.box(
    usage_plot,
    x="blinded_sample",
    y=PROGRAM,
    points=False,
    title=f"{LINEAGE} {PROGRAM}: distribution by blinded section",
)
figure.update_layout(template="plotly_white", showlegend=False)
figure

## High-usage cell audit

Review the cells that most strongly define the program. Their lineage labels, spatial domains, count depth, and detected-feature counts should be consistent with the proposed biological interpretation.

In [ ]:
high_usage = select_high_usage_cells(
    review_obs, program=PROGRAM, top_fraction=TOP_FRACTION
)
display(high_usage.head(50))

high_usage_counts = (
    high_usage.groupby(
        ["blinded_sample", "cluster_sub", "spatial_domain"],
        dropna=False,
        observed=True,
    )
    .size()
    .rename("n_high_usage_cells")
    .reset_index()
    .sort_values("n_high_usage_cells", ascending=False)
)
display(high_usage_counts.head(50))

## Expected-gene support

Compare raw-count detection and mean expression of the program's expected genes in high-usage cells versus other cells from the same cNMF lineage. This is a coherence check, not a differential-expression test.

In [ ]:
expected_genes = [gene for gene in str(program_row["top_genes"]).split("|") if gene]
gene_support = high_usage_gene_support(
    usage_adata, high_cell_ids=high_usage["cell_id"], genes=expected_genes
)
display(gene_support)
if gene_support.attrs.get("missing_genes"):
    print("Genes absent from this panel:", gene_support.attrs["missing_genes"])

## Spatial localization

Map program usage within a blinded tissue section. Look for anatomically coherent regions or multicellular patches and check for border effects, segmentation halos, or localization driven by only a few cells.

In [ ]:
high_section_counts = high_usage["blinded_sample"].value_counts()
shown_section = (
    str(high_section_counts.index[0])
    if SPATIAL_SECTION is None
    else str(SPATIAL_SECTION)
)
spatial_figure = plot_blinded_spatial_usage(
    review_obs,
    np.asarray(usage_adata.obsm["spatial"]),
    program=PROGRAM,
    blinded_sample=shown_section,
    point_size=SPATIAL_POINT_SIZE,
)
spatial_figure

## Neighboring-cell context

This table summarizes cell types near high-usage focal cells within each tissue section. Use it to distinguish an intrinsic lineage program from a pattern that may reflect neighboring-cell spillover.

In [ ]:
master = ad.read_h5ad(MASTER_H5AD, backed="r")
try:
    context_ids = high_usage["cell_id"].head(MAX_CONTEXT_FOCAL_CELLS).tolist()
    neighbor_context = high_usage_neighbor_context(
        master.obs[["sample_id", "cluster_sub"]],
        np.asarray(master.obsm["spatial"]),
        focal_cell_ids=context_ids,
        sample_key="sample_id",
        cell_type_key="cluster_sub",
        radius=NEIGHBOR_RADIUS_UM,
    )
finally:
    master.file.close()
display(neighbor_context.head(40))

## Relationship to other programs

Program correlations provide a condition-blind view of shared or opposing usage patterns within the lineage. Use them as descriptive context alongside genes and spatial evidence.

In [ ]:
correlation_obs = review_obs[programs]
if len(correlation_obs) > MAX_CORRELATION_CELLS:
    correlation_obs = correlation_obs.sample(MAX_CORRELATION_CELLS, random_state=0)

program_correlations = (
    correlation_obs.corr(method="spearman")[PROGRAM]
    .drop(PROGRAM)
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .rename_axis("other_program")
    .reset_index(name="spearman_correlation")
)
program_labels = whitelist.loc[
    whitelist["lineage"].eq(LINEAGE),
    ["program", "proposed_label", "decision"],
]
correlation_review = (
    program_correlations.merge(
        program_labels,
        left_on="other_program",
        right_on="program",
        how="left",
    )
    .drop(columns="program")
    .head(15)
)
display(correlation_review)

## Record a review decision

Assign one of four outcomes:

- **keep_primary** — coherent and suitable for the primary feature set;
- **keep_sensitivity** — plausible but better suited to sensitivity analysis;
- **exclude** — inconsistent, technical, mixed-lineage, or likely spillover;
- **needs_followup** — insufficient evidence for admission.

Add a short evidence-based rationale. Saving remains disabled until the explicit write switch is enabled.

In [ ]:
DRAFT_TO_REVIEW = {
    "include_primary": "keep_primary",
    "include_sensitivity": "keep_sensitivity",
    "exclude": "exclude",
}

REVIEW_DECISION = DRAFT_TO_REVIEW[str(program_row["decision"])]
REVIEW_CONFIDENCE = str(program_row["confidence"])
REVIEW_NOTES = ""
REVIEWER = ""
SAVE_REVIEW_DECISION = False

review_record = build_review_decision(
    lineage=LINEAGE,
    program=PROGRAM,
    review_decision=REVIEW_DECISION,
    review_confidence=REVIEW_CONFIDENCE,
    review_notes=REVIEW_NOTES,
    reviewer=REVIEWER,
    top_fraction=TOP_FRACTION,
    neighbor_radius_um=NEIGHBOR_RADIUS_UM,
    whitelist_sha256=whitelist_sha256,
)
display(pd.Series(review_record).to_frame("review_value"))

if SAVE_REVIEW_DECISION:
    saved_path = upsert_review_decision(DECISION_PATH, review_record)
    print("saved:", saved_path)
else:
    print("Not saved. Set SAVE_REVIEW_DECISION=True only after review.")

## Freeze the reviewed whitelist

Freeze the whitelist only after every admitted program has a saved decision, unresolved exclusions have been reviewed or deferred, and the decision table matches the current whitelist checksum.

The finalization script writes a versioned whitelist and manifest; only that frozen artifact should be used to construct downstream model features.